# Twisted-bilayer BEC of Meng et al. (Nature 615, 231, 2023): one cloud, five models

The same trapped cloud (⁸⁷Rb, V₀ = 4 E_R, 5.21° twist, 1 kHz pancake) computed in

| model | what it is |
|---|---|
| **full 3D GPE** | the reference: two coupled components on a 3D grid (x, y, z) |
| **NPSE (2D)** | quasi-2D GPE whose vertical width follows the local density (λ = 1) |
| **fixed g (2D)** | quasi-2D GPE with the bare pancake's interaction g₂D |
| **lattice GPE** | two-layer Bose–Hubbard model of the same box, coherent-state mean field |
| **Gutzwiller** | the same lattice model, single-site Gutzwiller (keeps number fluctuations) |

All five use the same trap and atom number, and every observable is computed by the same repository functions from
the column densities (the lattice models through their Wannier functions). The continuum models run in each box;
the lattice models run in the 3×3 box (a 5×5 lattice model needs more than 6 GB of RAM) and are padded with empty
sites for the 5×5 entries, which is exact because the cloud fits in 3×3.

* **C_coh**: the moiré contrast of the |2⟩ image of the central moiré cell as the paper's camera would record it
  (coherent absorption imaging, NA 0.69, saturation s); 1 = all atoms on the AA points, 0 = flat, < 0 = AB brighter.
* **n_AA, n_AB**: total density at the AA and AB points (atoms / a²); **n̄ centre**: mean filling of the central moiré cell.
* **R_rms**: cloud size; **μ − E_b**: chemical potential above the band bottom of one uncoupled lattice.

Run all cells. Results are cached in `CACHE_DIR` per (coupling, box, settings): rerunning with the same parameters
reads them back in a second; changing a parameter computes only what changed.

In [ ]:
# fmt: off
# ---- parameters: change these on the spot, then run the cells below ----
COUPLINGS   = (0.121, 0.150, 0.243)   # J⊥ in E_R: 0.150 = reproduces the 893 Hz Rabi fit, 0.243 = paper-literal; any value works
BOXES       = (3, 5)                  # box in √485 a supercells (odd); 3×3 = 66 a = 26 µm, 5×5 = 110 a = 43 µm
MODELS      = ("3d", "npse", "fixed", "lattice_gpe", "gutzwiller")

N_ATOMS     = 2000                    # atoms in the pancake (not published by the paper)
NU_R_HZ     = 20.0                    # radial trap frequency (not published)
V0_ER       = 4.0                     # lattice depth (E_R), the paper's Fig. 4
NU_Z_HZ     = 1000.0                  # pancake frequency

CELL_POINTS = 176                     # grid points per supercell (176 = 8 per a, converged)
NZ          = 16                      # vertical points of the 3D grid (±5 l_z)
DTYPE_3D    = "complex64"             # complex64 fits 5×5 in 6 GB; contrasts agree with complex128 to 4e-6
S_SAT       = 0.01                    # imaging saturation of C_coh

SHOW_J, SHOW_BOX = 0.150, 5           # the cloud shown in the image and cut panels
FORCE       = False                   # True: recompute even if cached
QUICK       = False                   # True: smoke test only (one coupling, one supercell, coarse grid; ~1 min on a CPU; numbers not physical)
CACHE_DIR   = "showcase_results"
# fmt: on

In [ ]:
import time

import matplotlib.pyplot as plt
import showcase_helpers as H
from IPython.display import Markdown, display

src = H.find_repo(".")
print("repository:", src)
print(H.device_report())

S = H.Settings(
    couplings=tuple(COUPLINGS),
    boxes=tuple(BOXES),
    models=tuple(MODELS),
    V0=V0_ER,
    N_atoms=float(N_ATOMS),
    nu_r_hz=NU_R_HZ,
    nu_z_hz=NU_Z_HZ,
    cell_points=CELL_POINTS,
    Nz=NZ,
    dtype_3d=DTYPE_3D,
    s_sat=S_SAT,
    cache_dir=CACHE_DIR,
)
if QUICK:
    S = H.Settings(
        couplings=(0.150,),
        boxes=(1,),
        models=tuple(MODELS),
        N_atoms=150.0,
        nu_r_hz=80.0,
        cell_points=64,
        Nz=8,
        cache_dir=CACHE_DIR + "_quick",
    )
    SHOW_J, SHOW_BOX = 0.150, 1
for b in S.boxes:
    print(
        f"{b}×{b}: grid {b * S.cell_points}² × {S.Nz}, 3D peak ≈ {H.memory_estimate_gib(S, b):.1f} GiB ({S.dtype_3d}); "
        f"lattice model {2 * 485 * b * b} sites"
    )

## Run

Cached pieces load instantly. From scratch on a GTX 1660 Ti: about 3–5 min (3×3) and 10–15 min (5×5) per coupling for
the three continuum models; the lattice models run on the CPU, about 5 min per coupling (once; the 5×5 entry reuses it).

In [ ]:
t0 = time.time()
R = H.run_all(S, force=FORCE)
print(f"all results in {time.time() - t0:.0f} s")

## 1. All numbers

In [ ]:
display(Markdown(H.table(R, S)))

## 2. The cloud in each model: column density
Both states together, the same colour scale for all models. The fine grid is the atomic lattice (a = 0.395 µm); the
brighter squares are the moiré pockets on the AA points (period 4.35 µm).

In [ ]:
fig = H.plot_gallery(R, S, SHOW_J, SHOW_BOX)
plt.show()

## 3. The two lattices: |1⟩ and |2⟩ in the central supercell
|1⟩ lives on lattice 1 (orange), |2⟩ on lattice 2 (blue), the two rotated by 5.2°. Where their sites coincide (AA, the
centre and the corners) both are occupied and the image turns white; in between (AB) they alternate.

In [ ]:
fig = H.plot_two_state(R, S, SHOW_J, SHOW_BOX)
plt.show()

## 4. What the camera sees
The |2⟩ image through the paper's objective (NA 0.69, coherent absorption imaging): the atomic lattice is blurred out,
the moiré pattern remains. C_coh, the contrast of this image's central moiré cell, is the number compared with Fig. 4.

In [ ]:
fig = H.plot_images(R, S, SHOW_J, SHOW_BOX, as_imaged=True)
plt.show()

## 5. Where each model departs from full 3D
Each model's column density minus 3D, averaged over one lattice site, in % of the peak density; each panel has its own
scale. Red: the model puts too many atoms there, blue: too few.

In [ ]:
fig = H.plot_differences(R, S, SHOW_J, SHOW_BOX)
if fig is not None:
    plt.show()

## 6. Radial profiles
Ring-averaged density against the distance from the centre: left site-averaged, right as the camera records it.

In [ ]:
fig = H.plot_radial(R, S, SHOW_J, SHOW_BOX)
plt.show()

## 7. Observables against the coupling
The coupling J⊥ is not settled by the paper (0.150 vs 0.243 E_R); this shows how much it matters compared with
the choice of model.

In [ ]:
for b in S.boxes:
    fig = H.plot_observables(R, S, b)
    plt.show()

## 8. How far each reduced model is from full 3D, in numbers
Relative deviation in %. The NPSE is the reduction we develop; fixed g is the common textbook choice;
the lattice models are where beyond-mean-field physics (Mott, cluster Gutzwiller, QMC) will be added.

In [ ]:
fig = H.plot_vs_3d(R, S, max(S.boxes))
plt.show()

## 9. Why the NPSE works: the vertical width
The pancake swells where the density is high; the NPSE follows it locally, fixed g ignores it.

In [ ]:
fig = H.plot_widths(R, SHOW_J, SHOW_BOX)
if fig is not None:
    plt.show()

## 10. Box truncation: the same cloud in the small and the large box

In [ ]:
fig = H.plot_boxes(R, S)
if fig is not None:
    plt.show()

## 11. In words

In [ ]:
big = max(S.boxes)
lines = []
for J in S.couplings:
    o3 = H.observables(R, J, big, "3d")
    if o3 is None:
        continue
    parts = [
        f"**J⊥ = {J} E_R** ({big}×{big}): 3D C_coh = {o3['C_coh']:.3f}, n̄ centre = {o3['nbar_centre_cell']:.2f}, "
        f"R_rms = {o3['R_rms'] * H.A_UM:.1f} µm"
    ]
    for m in S.models:
        if m == "3d":
            continue
        o = H.observables(R, J, big, m)
        if o is not None:
            parts.append(
                f"{H.NAMES[m]} ΔC_coh = {o['C_coh'] - o3['C_coh']:+.4f}, Δn_AA = {100 * (o['n_AA'] - o3['n_AA']) / o3['n_AA']:+.1f} %"
            )
    lines.append("- " + "; ".join(parts))
    if len(S.boxes) > 1:
        a = H.observables(R, J, min(S.boxes), "3d")
        if a is not None:
            lines.append(
                f"  - box truncation of 3D C_coh: {a['C_coh'] - o3['C_coh']:+.2e} "
                f"({min(S.boxes)}×{min(S.boxes)} − {big}×{big})"
            )
display(Markdown("\n".join(lines)))

## 12. Save the figures

In [ ]:
import pathlib

out = pathlib.Path(S.cache_dir) / "figures"
out.mkdir(parents=True, exist_ok=True)
figs = {
    "gallery": H.plot_gallery(R, S, SHOW_J, SHOW_BOX),
    "two_state": H.plot_two_state(R, S, SHOW_J, SHOW_BOX),
    "camera": H.plot_images(R, S, SHOW_J, SHOW_BOX, as_imaged=True),
    "differences": H.plot_differences(R, S, SHOW_J, SHOW_BOX),
    "radial": H.plot_radial(R, S, SHOW_J, SHOW_BOX),
    "vs_3d": H.plot_vs_3d(R, S, max(S.boxes)),
    "widths": H.plot_widths(R, SHOW_J, SHOW_BOX),
    "boxes": H.plot_boxes(R, S),
}
figs.update({f"observables_{b}x{b}": H.plot_observables(R, S, b) for b in S.boxes})
for k, f in figs.items():
    if f is not None:
        f.savefig(out / f"{k}.png", dpi=170, facecolor=f.get_facecolor())
        plt.close(f)
(out.parent / "table.md").write_text(H.table(R, S), encoding="utf-8")
print("saved to", out.resolve())